# IIR Low-Pass Filter on IQ Signal: Plain Instructions → Code

**Prompt source:** `prompts/iir-plain-instructions-to-code.md`

Implements an IIR low-pass filter on a synthetic IQ signal following natural-language requirements:

1. Generate synthetic IQ signal — sinusoidal + noise, shape `(N=5, 2, L=1000)`
2. Design 2nd-order Butterworth low-pass filter (`cutoff=0.1`, normalised)
3. Apply filter along `axis=2` (time) for I and Q separately
4. Compute power before and after
5. Plot: I trace, Q trace, constellation, power comparison
6. Save filtered signal to `filtered_iq.npz`
7. Print verification: shape, dtype, power before/after, ratio

## 1. Imports

In [ ]:
import numpy as np
from scipy.signal import butter, lfilter
import matplotlib.pyplot as plt

## 2. Synthetic IQ Signal Generation

Shape convention: `(N, 2, L)` — axis 0 = examples, axis 1 = [I, Q], axis 2 = time samples.

Each example is a sinusoidal carrier at a slightly different frequency plus Gaussian noise,
simulating a realistic IQ signal.

In [ ]:
# Parameters
N    = 5
L    = 1000
SEED = 42

rng = np.random.default_rng(SEED)
t   = np.linspace(0, 1, L, endpoint=False)  # time axis

# Build (N, 2, L) tensor: sinusoidal + noise per example
X = np.zeros((N, 2, L), dtype=np.float32)
for n in range(N):
    freq       = 0.05 + n * 0.04          # different freq per example (5%, 9%, 13%, 17%, 21% of Fs)
    noise_std  = 0.5
    X[n, 0, :] = (np.cos(2 * np.pi * freq * L * t)          # I: cosine carrier
                  + noise_std * rng.standard_normal(L)).astype(np.float32)
    X[n, 1, :] = (np.sin(2 * np.pi * freq * L * t)          # Q: sine carrier
                  + noise_std * rng.standard_normal(L)).astype(np.float32)

print(f'X shape : {X.shape}   (N, 2, L)')
print(f'X dtype : {X.dtype}')
print(f'I mean  : {X[:, 0, :].mean():.4f}   Q mean: {X[:, 1, :].mean():.4f}')

## 3. IIR Filter Design

2nd-order Butterworth low-pass with normalised cutoff `0.1` (= 10% of Nyquist).

In [ ]:
CUTOFF = 0.1   # normalised frequency (fraction of Nyquist)
ORDER  = 2

b, a = butter(ORDER, CUTOFF, btype='low')

print(f'Filter order : {ORDER}')
print(f'Cutoff (norm): {CUTOFF}')
print(f'b coeffs     : {np.round(b, 6)}')
print(f'a coeffs     : {np.round(a, 6)}')

## 4. Apply Filter Along Time Axis

`lfilter` with `axis=2` filters each I and Q component independently across time samples.

In [ ]:
X_filtered = lfilter(b, a, X, axis=2).astype(np.float32)

assert X_filtered.shape == X.shape, \
    f'Shape mismatch: {X_filtered.shape} vs {X.shape}'

print(f'X_filtered shape : {X_filtered.shape}   — shape preserved ✓')
print(f'X_filtered dtype : {X_filtered.dtype}')

## 5. Power Before and After Filtering

Power per example: `P = mean(I² + Q²)`

In [ ]:
def compute_power(X: np.ndarray) -> float:
    """Mean power across all examples and time samples. X shape: (N, 2, L)."""
    return float(np.mean(X[:, 0, :]**2 + X[:, 1, :]**2))

p_before = compute_power(X)
p_after  = compute_power(X_filtered)
ratio_db = 10 * np.log10(p_after / p_before)

print('─── Verification ────────────────────────────')
print(f'Shape  before : {X.shape}  dtype={X.dtype}')
print(f'Shape  after  : {X_filtered.shape}  dtype={X_filtered.dtype}')
print(f'Power  before : {p_before:.6f}')
print(f'Power  after  : {p_after:.6f}')
print(f'Power ratio   : {ratio_db:.3f} dB')
print('─────────────────────────────────────────────')

assert p_after < p_before, 'Expected power reduction after low-pass filtering'
print('Power reduction assert PASSED ✓')

## 6. Visualization — 4 Subplots

(a) I time trace · (b) Q time trace · (c) I/Q constellation · (d) Power comparison

In [ ]:
EXAMPLE = 0   # which example to plot

I_raw  = X[EXAMPLE, 0, :]
Q_raw  = X[EXAMPLE, 1, :]
I_filt = X_filtered[EXAMPLE, 0, :]
Q_filt = X_filtered[EXAMPLE, 1, :]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'IIR Low-Pass Filter on IQ Signal — Example {EXAMPLE}', fontsize=13)

# (a) I time trace
ax = axes[0, 0]
ax.plot(I_raw,  color='steelblue', linewidth=0.7, alpha=0.8, label='Raw')
ax.plot(I_filt, color='tomato',    linewidth=1.2,             label='Filtered')
ax.set_title('(a) I Channel — Time Trace')
ax.set_xlabel('Sample')
ax.set_ylabel('Amplitude')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Q time trace
ax = axes[0, 1]
ax.plot(Q_raw,  color='steelblue', linewidth=0.7, alpha=0.8, label='Raw')
ax.plot(Q_filt, color='tomato',    linewidth=1.2,             label='Filtered')
ax.set_title('(b) Q Channel — Time Trace')
ax.set_xlabel('Sample')
ax.set_ylabel('Amplitude')
ax.legend()
ax.grid(True, alpha=0.3)

# (c) I/Q constellation
ax = axes[1, 0]
ax.scatter(I_raw,  Q_raw,  s=2, color='steelblue', alpha=0.5, label='Raw')
ax.scatter(I_filt, Q_filt, s=4, color='tomato',    alpha=0.8, label='Filtered')
ax.set_title('(c) I/Q Constellation')
ax.set_xlabel('I')
ax.set_ylabel('Q')
ax.legend(markerscale=4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# (d) Power comparison per example
ax = axes[1, 1]
p_before_per = [compute_power(X[n:n+1]) for n in range(N)]
p_after_per  = [compute_power(X_filtered[n:n+1]) for n in range(N)]
x_idx = np.arange(N)
width = 0.35
ax.bar(x_idx - width/2, p_before_per, width, color='steelblue', alpha=0.8, label='Before')
ax.bar(x_idx + width/2, p_after_per,  width, color='tomato',    alpha=0.8, label='After')
ax.set_title('(d) Power per Example')
ax.set_xlabel('Example index')
ax.set_ylabel('Mean Power')
ax.set_xticks(x_idx)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Save Filtered Signal to NPZ

In [ ]:
np.savez(
    'filtered_iq.npz',
    X_raw=X,
    X_filtered=X_filtered,
    b=b,
    a=a,
    p_before=np.array(p_before),
    p_after=np.array(p_after)
)
print('Saved → filtered_iq.npz')

# Verify round-trip
data = np.load('filtered_iq.npz')
assert data['X_filtered'].shape == (N, 2, L)
print(f'Round-trip shape : {data["X_filtered"].shape} ✓')
print(f'Keys in file     : {list(data.keys())}')

---
## Summary

| Step | Detail |
|------|--------|
| Signal | Sinusoidal + noise, shape `(5, 2, 1000)`, `float32`, `SEED=42` |
| Filter | 2nd-order Butterworth low-pass, `cutoff=0.1` (normalised) |
| Application | `scipy.signal.lfilter(b, a, X, axis=2)` |
| Power | Reduced after filtering (`p_after < p_before`) |
| Output | `filtered_iq.npz` with keys: `X_raw`, `X_filtered`, `b`, `a`, `p_before`, `p_after` |

**Next steps:**
- Try `sosfilt` instead of `lfilter` for better numerical stability
- Vary `CUTOFF` (0.05–0.4) to observe filter aggressiveness
- Load `filtered_iq.npz` into the GNU Radio notebook for further processing